In [21]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 


In [23]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# as_of = datetime.date(2026, 2, 23)
# start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
# end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

start = NY_tz.localize(datetime.datetime(2026, 2, 1, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 2, 20, 23, 59))

# mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
# pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

MERGING SLICES...: 100%|██████████| 14/14 [00:00<00:00, 21.59it/s]


In [27]:
upis = pd.read_csv(r"C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Rates-Option-CapFloor.csv")["Identifier_UPI"]
upis

0       QZQ2G06W2635
1       QZN5GRNTN647
2       QZ5NW1PZBSX6
3       QZVB7HGNLT3D
4       QZRC3MD135GC
            ...     
3684    QZSJSJ6L00K5
3685    QZDNCVL0KVGW
3686    QZ24HVSWLMJ9
3687    QZJ1DDT7L86C
3688    QZ83XT5J5JBQ
Name: Identifier_UPI, Length: 3689, dtype: object

In [29]:
df[df["Unique Product Identifier"].isin(upis)]["Unique Product Identifier"].value_counts()

# df[(df["Effective Date"].dt.date == datetime.date(2026, 6, 17)) & (df["Expiration Date"].dt.date == datetime.date(2026, 7, 29))].to_csv("june_fomc_dated_sdr_trades.csv")

Unique Product Identifier
QZXNP136XML0    435
QZQJWDQ4V0VJ    336
QZ7VKKVLNLP0    213
QZL9GWT1VKHZ    133
QZTSC5QX8B68    112
               ... 
QZFG37ZLHPC2      1
QZMKMFN3BQ64      1
QZKB3NH4Z39F      1
QZ2ZTJ0F8Q1N      1
QZ8ZJN6707R1      1
Name: count, Length: 113, dtype: int64

In [33]:
df[df["Unique Product Identifier"] == "QZXNP136XML0"].iloc[1].to_dict()

{'Dissemination Identifier': '1931696897000000401',
 'Original Dissemination Identifier': '',
 'Action type': 'NEWT',
 'Event type': 'TRAD',
 'Event timestamp': Timestamp('2026-02-02 15:13:07+0000', tz='UTC'),
 'Amendment indicator': None,
 'Asset Class': 'IR',
 'Product name': None,
 'Cleared': 'N',
 'Mandatory clearing indicator': False,
 'Execution Timestamp': Timestamp('2025-03-28 13:20:44+0000', tz='UTC'),
 'Effective Date': Timestamp('2025-03-28 00:00:00'),
 'Expiration Date': Timestamp('2027-01-06 00:00:00'),
 'Maturity date of the underlier': None,
 'Non-standardized term indicator': False,
 'Platform identifier': 'BILT',
 'Prime brokerage transaction indicator': False,
 'Block trade election indicator': False,
 'Large notional off-facility swap election indicator': False,
 'Notional amount-Leg 1': '53,000,000',
 'Notional amount-Leg 2': '',
 'Notional currency-Leg 1': 'USD',
 'Notional currency-Leg 2': '',
 'Notional quantity-Leg 1': None,
 'Notional quantity-Leg 2': None,
 'T

In [ ]:
# sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
sdf = USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=False, merge_package_legs=False)
sdf

In [7]:
sdf.to_csv("usd_swaps_sdr_classification_data.csv",index=False)